# MVP: Classificação de Risco de Saúde Materna

## Introdução e Definição do Problema
O objetivo deste projeto é construir um modelo de Machine Learning capaz de classificar o nível de risco de gestantes (Baixo, Médio ou Alto Risco) com base em métricas fisiológicas e sinais vitais, como idade, pressão arterial, níveis de glicose no sangue, temperatura corporal e frequência cardíaca.

## Importação das bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib



##Carga do Dataset
O dataset contém informações sobre gestantes, com cada registo a representar uma paciente. As variáveis incluem características fisiológicas e sinais vitais. A variável alvo (target) é o `RiskLevel`, que classifica o risco da gravidez.

**Descrição de Cada Coluna:**
- **Age:** Idade da paciente (anos).
- **SystolicBP:** Pressão arterial sistólica (mmHg).
- **DiastolicBP:** Pressão arterial diastólica (mmHg).
- **BS:** Níveis de glicose no sangue (mmol/L).
- **BodyTemp:** Temperatura corporal (Fahrenheit).
- **HeartRate:** Frequência cardíaca (batimentos por minuto).
- **RiskLevel:** Nível de risco da gravidez (Baixo, Médio ou Alto).

In [ ]:
# Carga do dataset via URL Raw do GitHub (Repositório da Brenda)
url_dataset = 'https://raw.githubusercontent.com/Brendalisboa/Maternal-Health-Risk/main/Maternal_Risk.csv'
df = pd.read_csv(url_dataset)

# Exibindo as primeiras linhas para confirmar a carga
display(df.head())

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


## Separação de Dados e Escolha dos Algoritmos
Para garantir que nosso modelo consiga generalizar bem para novas pacientes, utilizaremos a técnica de Holdout, separando 20% dos dados para teste e 80% para treinamento. Em seguida, configuraremos Pipelines para avaliar **quatro algoritmos distintos**, conforme os requisitos do projeto:
1. **KNN (K-Nearest Neighbors):** Necessita de padronização (`StandardScaler`) pois é baseado em cálculo de distâncias.
2. **CART (Árvore de Classificação):** Cria regras de divisão (splits) baseadas nas features. Não é estritamente sensível à escala dos dados.
3. **Naive Bayes (GaussianNB):** Algoritmo probabilístico baseado no Teorema de Bayes.
4. **SVM (Support Vector Machine):** Busca o hiperplano de separação máxima; utilizaremos `MinMaxScaler` para normalizar os dados e otimizar a convergência.

Utilizaremos Validação Cruzada (Cross-Validation com 10 partições) para garantir uma avaliação justa da acurácia média de cada algoritmo no conjunto de treino.

In [ ]:
# Separação de features (X) e target (y)
X = df.drop('RiskLevel', axis=1)
y = df['RiskLevel']

# Separação Holdout (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Configuração da Validação Cruzada (10 folds)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Construção dos Pipelines com os 4 algoritmos obrigatórios
pipelines = []
pipelines.append(('KNN-Pad', Pipeline([('Scaler', StandardScaler()), ('KNN', KNeighborsClassifier())])))
pipelines.append(('CART', Pipeline([('CART', DecisionTreeClassifier(random_state=42))])))
pipelines.append(('NB', Pipeline([('NB', GaussianNB())])))
pipelines.append(('SVM-Norm', Pipeline([('Scaler', MinMaxScaler()), ('SVM', SVC(random_state=42))])))

# Execução da Validação Cruzada
print("Comparação de Algoritmos - Acurácia Média na Validação Cruzada (10 folds):\n")
for nome, modelo in pipelines:
    cv_results = cross_val_score(modelo, X_train, y_train, cv=kfold, scoring='accuracy')
    print(f"{nome}: {cv_results.mean():.4f} (Desvio Padrão: {cv_results.std():.4f})")

Comparação de Algoritmos - Acurácia Média na Validação Cruzada (10 folds):

KNN-Pad: 0.9180 (Desvio Padrão: 0.0456)
CART: 0.9566 (Desvio Padrão: 0.0275)
NB: 0.8871 (Desvio Padrão: 0.0571)
SVM-Norm: 0.8931 (Desvio Padrão: 0.0606)


## Otimização de Hiperparâmetros (O Vencedor: CART)
Como observado nos resultados acima, a **Árvore de Classificação (CART)** obteve o melhor desempenho na validação cruzada. Por isso, ela foi a escolhida para a etapa de otimização de hiperparâmetros via `GridSearchCV`.

**O que significam os hiperparâmetros no contexto do risco gestacional?**
* **`criterion` (gini ou entropy):** Define a função matemática que mede a qualidade de uma divisão (split). A *entropia*, por exemplo, busca minimizar a "desordem", garantindo que as folhas da árvore tenham pacientes com níveis de risco o mais puros e homogêneos possível.
* **`max_depth`:** Define a profundidade máxima da árvore (quantas perguntas sequenciais o modelo fará, ex: "Idade > 35?" -> "Pressão > 140?"). Limitar isso (ex: a 10) evita que o modelo "decore" o conjunto de treino (overfitting) e crie regras muito específicas para apenas uma paciente.
* **`min_samples_split`:** O número mínimo de pacientes necessárias em um nó para que ele seja dividido.

Vamos buscar a melhor combinação destes parâmetros e avaliar o modelo no nosso conjunto de teste isolado.

In [ ]:
# Definindo a grade de hiperparâmetros para o modelo CART (índice 1 na lista de pipelines)
param_grid_cart = {
    'CART__max_depth': [None, 5, 10, 15, 20],
    'CART__min_samples_split': [2, 5, 10],
    'CART__criterion': ['gini', 'entropy']
}

# Realizando a busca exaustiva (GridSearchCV)
grid_search = GridSearchCV(pipelines[1][1], param_grid_cart, cv=kfold, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Melhores hiperparâmetros encontrados para CART: {grid_search.best_params_}\n")

# Coletando o melhor modelo e testando com dados não vistos (Holdout)
melhor_modelo = grid_search.best_estimator_
predicoes = melhor_modelo.predict(X_test)
acuracia_teste = accuracy_score(y_test, predicoes)

print(f"Acurácia final no conjunto de Teste: {acuracia_teste:.4f}")

# Exportação do modelo para o Back-end
joblib.dump(melhor_modelo, 'modelo_risco_materno_cart.pkl')
print("\nModelo exportado com sucesso para 'modelo_risco_materno_cart.pkl'!")

Melhores hiperparâmetros encontrados para CART: {'CART__criterion': 'entropy', 'CART__max_depth': 10, 'CART__min_samples_split': 2}

Acurácia final no conjunto de Teste: 0.9815

Modelo exportado com sucesso para 'modelo_risco_materno_cart.pkl'!


In [ ]:
import pandas as pd

# Transformando as variáveis de teste (Holdout) em DataFrames do Pandas
X_test_df = pd.DataFrame(X_test, columns=X.columns)
y_test_df = pd.DataFrame(y_test, columns=['RiskLevel'])

# Exportando para arquivos CSV (sem a coluna de índice)
X_test_df.to_csv('X_test_risco_materno.csv', index=False)
y_test_df.to_csv('y_test_risco_materno.csv', index=False)

print("Arquivos CSV gerados com sucesso!")

Arquivos CSV gerados com sucesso!


## Análise de Resultados e Conclusão

**Resumo dos Achados e Desempenho:**
O desenvolvimento deste modelo preditivo demonstrou que as variáveis fisiológicas coletadas são excelentes preditoras do risco gestacional. Na comparação inicial, a **Árvore de Classificação (CART)** superou os modelos KNN, Naive Bayes e SVM. Após a otimização com hiperparâmetros restritivos (como `max_depth=10` e `criterion='entropy'`), o modelo alcançou uma acurácia excepcional no conjunto de teste isolado (holdout). Isso comprova que o modelo conseguiu generalizar as regras médicas intrínsecas aos dados sem sofrer *overfitting*.

**Pontos de Atenção (Caveats):**
Apesar da alta métrica global, no contexto da saúde, falsos negativos (classificar um risco alto como baixo) são mais perigosos do que falsos positivos. Em iterações futuras deste MVP, seria fundamental analisar métricas complementares como o *Recall* (Revocação) da classe minoritária ("High Risk") para assegurar que a taxa de falsos seguros seja minimizada a quase zero.

**Fechamento e Próximos Passos:**
O pipeline atingiu seu propósito central: carregar os dados, pré-processá-los, validar diferentes arquiteturas, otimizar a vencedora de forma justificada e empacotar o conhecimento gerado. Com o arquivo `modelo_risco_materno_cart.pkl` exportado com sucesso, o projeto está pronto para sua fase final. O modelo será embarcado na API do Back-end (Flask) e consumido por um Front-end intuitivo, cumprindo integralmente os requisitos de uma aplicação Full Stack orientada a dados.